# 06 — Hyperparameter Tuning & Generalization Check

**Goal:** Fine-tune our models and test if they are Overfitting or Underfitting.

## 1. Imports & Data Loading

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import joblib
import warnings
warnings.filterwarnings('ignore')

FEATURED_PATH = '../data/processed/featured.csv'
MODELS_DIR = '../models/'

df = pd.read_csv(FEATURED_PATH, parse_dates=['datetime'], index_col='datetime')
TARGET = 'Global_active_power'
FEATURES = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend', 
            'lag_1h', 'lag_24h', 'lag_48h', 'rolling_mean_6h', 'rolling_mean_24h', 'unmetered_energy']

split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

Train shape: (13514, 11), Test shape: (3379, 11)


## 2. TimeSeriesSplit

In [2]:
tscv = TimeSeriesSplit(n_splits=3)

## 3. Tuning Ridge Regression

In [3]:
print('Tuning Ridge...')
ridge_search = RandomizedSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}, n_iter=4, cv=tscv, scoring='neg_mean_absolute_error', random_state=42)
ridge_search.fit(X_train, y_train)
best_ridge = ridge_search.best_estimator_

Tuning Ridge...


## 4. Tuning Random Forest

In [4]:
print('Tuning RF...')
rf_search = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), {'n_estimators': [50, 100], 'max_depth': [5, 10, 15]}, n_iter=4, cv=tscv, scoring='neg_mean_absolute_error', random_state=42)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

Tuning RF...


## 5. Tuning XGBoost

In [5]:
print('Tuning XGBoost...')
xgb_search = RandomizedSearchCV(XGBRegressor(random_state=42), {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5, 7]}, n_iter=5, cv=tscv, scoring='neg_mean_absolute_error', random_state=42)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

Tuning XGBoost...


## 6. Overfitting vs Underfitting Analysis

**What:** Pitting all models against BOTH the Train set and the Test set.
**Why:** This is the ultimate health check.
- **Overfitting:** Train score is perfect, Test score is terrible (Memorized the past, failed the future).
- **Underfitting:** Train score is terrible, Test score is terrible (Failed to learn anything).
- **Generalized:** Train score is good, Test score is close to Train score (Healthy).

In [6]:
def check_generalization(model, name):
    train_preds = model.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_preds)
    train_r2 = r2_score(y_train, train_preds)
    
    test_preds = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_preds)
    test_r2 = r2_score(y_test, test_preds)
    
    print(f'\n--- {name} ---')
    print(f'TRAIN -> MAE: {train_mae:.4f} kW | R²: {train_r2:.4f}')
    print(f'TEST  -> MAE: {test_mae:.4f} kW | R²: {test_r2:.4f}')
    
    if train_r2 > 0.90 and test_r2 < 0.60:
        print('Status: OVERFITTING (Memorized the past, failed the future)')
    elif train_r2 < 0.50 and test_r2 < 0.50:
        print('Status: UNDERFITTING (Too simple, failed to learn the pattern)')
    else:
        diff = test_mae - train_mae
        print(f'Status: GENERALIZED (Healthy. Test error is {diff:.4f} kW worse than Train)')
    return test_mae

ridge_mae = check_generalization(best_ridge, 'Tuned Ridge')
rf_mae = check_generalization(best_rf, 'Tuned Random Forest')
xgb_mae = check_generalization(best_xgb, 'Tuned XGBoost')

# Dynamically save the best
models_dict = {'Ridge': (best_ridge, ridge_mae), 'Random Forest': (best_rf, rf_mae), 'XGBoost': (best_xgb, xgb_mae)}
winner_name = min(models_dict, key=lambda k: models_dict[k][1])
winner_model = models_dict[winner_name][0]
joblib.dump(winner_model, f'{MODELS_DIR}best_model.pkl')
print(f'\nWinner: {winner_name}! Saved as best_model.pkl')


--- Tuned Ridge ---
TRAIN -> MAE: 0.3589 kW | R²: 0.7400
TEST  -> MAE: 0.2894 kW | R²: 0.7504
Status: GENERALIZED (Healthy. Test error is -0.0695 kW worse than Train)

--- Tuned Random Forest ---
TRAIN -> MAE: 0.2358 kW | R²: 0.8799
TEST  -> MAE: 0.2383 kW | R²: 0.7790
Status: GENERALIZED (Healthy. Test error is 0.0025 kW worse than Train)

--- Tuned XGBoost ---
TRAIN -> MAE: 0.2854 kW | R²: 0.8208
TEST  -> MAE: 0.2330 kW | R²: 0.7910
Status: GENERALIZED (Healthy. Test error is -0.0524 kW worse than Train)

Winner: XGBoost! Saved as best_model.pkl
